## threading

In [67]:
import threading
import time
import queue
import socket

import requests

In [24]:
%%time


N = 20
N_THREADS = 10
N_JOBS = N // N_THREADS

URL = "https://www.python.org/"
URLS = [URL] * N


def process_error(*args, **kwargs):
    print(f"process_error name={threading.current_thread().name}, {args=}, {kwargs=}")


threading.excepthook = process_error


def fetch_urls(que):
    print(f"fetch_urls, name={threading.current_thread().name} -- started")
    t1 = time.time()
    cnt = 0

    while True:
        url = que.get()
        if url is None:
            que.put(url)
            break

        resp = requests.get(url)
        cnt += 1
        if cnt == 3:
            raise ValueError(f"Wrong cnt={cnt}")

    t2 = time.time()
    print(f"fetch_urls, name={threading.current_thread().name} -- finished", t2 - t1, cnt)


def run():
    que = queue.Queue()
    for url in URLS:
        que.put(url)
    que.put(None)

    threads = [
        threading.Thread(
            target=fetch_urls,
            args=(que,),
            name=f"th_counter_{i}",
        )
        for i in range(N_THREADS)
    ]
    
    for th in threads:
        th.start()
    
    for th in threads:
        th.join()

run()

print("finished")

fetch_urls, name=th_counter_0 -- startedfetch_urls, name=th_counter_1 -- started

fetch_urls, name=th_counter_2 -- started
fetch_urls, name=th_counter_3 -- started
fetch_urls, name=th_counter_4 -- started
fetch_urls, name=th_counter_5 -- started
fetch_urls, name=th_counter_6 -- started
fetch_urls, name=th_counter_7 -- started
fetch_urls, name=th_counter_8 -- started
fetch_urls, name=th_counter_9 -- started
fetch_urls, name=th_counter_3 -- finished 1.2101237773895264 2
fetch_urls, name=th_counter_2 -- finished 1.2138111591339111 2
fetch_urls, name=th_counter_0 -- finished 1.465479850769043 2
fetch_urls, name=th_counter_7 -- finished 1.477903127670288 2
fetch_urls, name=th_counter_9 -- finished 1.5619268417358398 2
fetch_urls, name=th_counter_5 -- finished 1.5913670063018799 2
fetch_urls, name=th_counter_8 -- finished 1.6041536331176758 2
fetch_urls, name=th_counter_6 -- finished 1.6146249771118164 2
fetch_urls, name=th_counter_1 -- finished 1.7706129550933838 1
process_error name=th_cou

In [25]:
def process_data(data, event):
    print(f"process_data {data=} -- start")

    event.wait()

    print(f"process_data {data=} -- finish")


event = threading.Event()


th1 = threading.Thread(target=process_data, args=("th1", event))
th2 = threading.Thread(target=process_data, args=("th2", event))

print("init ---")
th1.start()
th2.start()
print("start --")

init ---
process_data data='th1' -- start
process_data data='th2' -- start
start --
process_data data='th2' -- finish
process_data data='th1' -- finish


In [26]:
th1.is_alive()

True

In [27]:
event.is_set()

False

In [28]:
event.set()

In [29]:
event.is_set()

True

## Process

In [35]:
import multiprocessing as mp
import os

In [31]:
mp.cpu_count()

12

In [32]:
mp.active_children()

[]

In [33]:
mp.get_all_start_methods()

['spawn', 'fork', 'forkserver']

In [34]:
mp.get_start_method()

'spawn'

In [36]:
os.getpid(), os.getppid()

(35355, 34639)

In [38]:
def process_data(data):
    print(f"process_data {data=} -- start")

    time.sleep(data)

    print(f"process_data {data=} -- finish")


pr = mp.Process(target=process_data, args=(10,))
pr.start()
pr.join()

Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=89, pipe_handle=99)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: module '__main__' has no attribute 'process_data'


In [39]:
ctx = mp.get_context("fork")

In [44]:
def process_data(data):
    print(f"process_data {data=} -- start")

    time.sleep(data)

    print(f"process_data {data=} -- finish")


pr = ctx.Process(target=process_data, args=(10,), name="Pr_process")
pr.start()
print(f"{pr.is_alive()=}, {pr.name=}, {pr.daemon=}, {pr.pid=}")
pr.join()

process_data data=10 -- start
pr.is_alive()=True, pr.name='Pr_process', pr.daemon=False, pr.pid=8945
process_data data=10 -- finish


In [47]:
pr.is_alive(), pr.name, pr.daemon, pr.pid

(False, 'Pr_process', False, 8945)

In [52]:
%%time

N = 10 ** 9
N_PROCS = 10
N_JOBS = N // N_PROCS


def counter(a, b, lock):
    with lock:
        print(f"counter[{a}, {b}], {os.getpid()=} -- started")

    while a < b:
        a += 1

    with lock:
        print(f"counter[{a}, {b}], {os.getpid()=} -- finished")


def run():
    lock = ctx.Lock()

    processes = [
        ctx.Process(
            target=counter,
            args=(i * N_JOBS, (i + 1) * N_JOBS, lock),
            name=f"pr_counter_{i}",
        )
        for i in range(N_PROCS)
    ]
    
    for pr in processes:
        pr.start()
    
    for pr in processes:
        pr.join()


run()

print("finished")

counter[0, 100000000], os.getpid()=38487 -- started
counter[100000000, 200000000], os.getpid()=38488 -- started
counter[200000000, 300000000], os.getpid()=38489 -- started
counter[300000000, 400000000], os.getpid()=38490 -- started
counter[400000000, 500000000], os.getpid()=38491 -- started
counter[500000000, 600000000], os.getpid()=38492 -- started
counter[600000000, 700000000], os.getpid()=38493 -- started
counter[700000000, 800000000], os.getpid()=38494 -- started
counter[800000000, 900000000], os.getpid()=38495 -- started
counter[900000000, 1000000000], os.getpid()=38496 -- started
counter[400000000, 400000000], os.getpid()=38490 -- finished
counter[200000000, 200000000], os.getpid()=38488 -- finished
counter[600000000, 600000000], os.getpid()=38492 -- finished
counter[700000000, 700000000], os.getpid()=38493 -- finished
counter[500000000, 500000000], os.getpid()=38491 -- finished
counter[300000000, 300000000], os.getpid()=38489 -- finished
counter[800000000, 800000000], os.getpid(

In [65]:
class User:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return f"User[{self.name},id={id(self)}]"


def update_data(data):
    print(f"start  update_data {data=}, {os.getpid()=}", data)

    data[123] = "123"
    data["dct"] = {1: 11, 2: 22}
    data[(1, "2", (3,))] = User("steve")
    data["to_update"] = "updated"

    print(f"finish update_data {data=}, {os.getpid()=}", data)

In [57]:
data = {"init": "initial_val", "to_update": "None"}

update_data(data)

print("after update", data)

start  update_data data={'init': 'initial_val', 'to_update': 'None'}, os.getpid()=35355
finish update_data data={'init': 'initial_val', 'to_update': 'updated', 123: '123', 'dct': {1: 11, 2: 22}, (1, '2', (3,)): User[steve,id=4478860048]}, os.getpid()=35355
after update {'init': 'initial_val', 'to_update': 'updated', 123: '123', 'dct': {1: 11, 2: 22}, (1, '2', (3,)): User[steve,id=4478860048]}


In [59]:
data = {"init": "initial_val", "to_update": None}


pr = ctx.Process(target=update_data, args=(data,))
pr.start()
pr.join()

print("after update", data)

start  update_data data={'init': 'initial_val', 'to_update': None}, os.getpid()=60581
finish update_data data={'init': 'initial_val', 'to_update': 'updated', 123: '123', 'dct': {1: 11, 2: 22}, (1, '2', (3,)): User[steve,id=4506340880]}, os.getpid()=60581
after update {'init': 'initial_val', 'to_update': None}


In [66]:
#data = {"init": "initial_val", "to_update": None}


with ctx.Manager() as manager:
    local_data = manager.dict()
    local_data["init"] = "initial_val"
    local_data["to_update"] = None

    pr = ctx.Process(target=update_data, args=(local_data,))
    pr.start()
    pr.join()

    print("local_data after update", local_data)

    data = dict(local_data)

print("data after update", data)

start  update_data data=<DictProxy object, typeid 'dict' at 0x10dba89d0>, os.getpid()=98181 {'init': 'initial_val', 'to_update': None}
finish update_data data=<DictProxy object, typeid 'dict' at 0x10dba89d0>, os.getpid()=98181 {'init': 'initial_val', 'to_update': 'updated', 123: '123', 'dct': {1: 11, 2: 22}, (1, '2', (3,)): User[steve,id=4518964464]}
local_data after update {'init': 'initial_val', 'to_update': 'updated', 123: '123', 'dct': {1: 11, 2: 22}, (1, '2', (3,)): User[steve,id=4518964464]}
data after update {'init': 'initial_val', 'to_update': 'updated', 123: '123', 'dct': {1: 11, 2: 22}, (1, '2', (3,)): User[steve,id=4518963456]}


In [77]:
def client(data):
    print(f"start client {data=}")
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.connect(("localhost", 20_001))
        sock.sendall(data.encode())
        updated = sock.recv(1024).decode()
        print(f"client recv {updated=}")

    print(f"finish client {data=}")


def server():
    print(f"start server")

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("localhost", 20_001))
        sock.listen(5)

        while True:
            client, addr = sock.accept()
            print(f"server accepted {client=}, {addr=}")

            data = client.recv(1024)
            data = data.decode()
            print(f"recv {data=}")

            client.sendall(data.upper().encode())

    print(f"finish server")


pr_server = ctx.Process(target=server)
pr_server.start()

start server
server accepted client=<socket.socket fd=26, family=2, type=1, proto=0, laddr=('127.0.0.1', 20001), raddr=('127.0.0.1', 57323)>, addr=('127.0.0.1', 57323)
recv data='данные'
server accepted client=<socket.socket fd=36, family=2, type=1, proto=0, laddr=('127.0.0.1', 20001), raddr=('127.0.0.1', 57339)>, addr=('127.0.0.1', 57339)
recv data='qwerty'


In [78]:
pr_server.is_alive()

True

In [79]:
pr_client = ctx.Process(target=client, args=("данные",))
pr_client.start()


start client data='данные'
client recv updated='ДАННЫЕ'
finish client data='данные'


In [80]:
client("qwerty")

start client data='qwerty'
client recv updated='QWERTY'
finish client data='qwerty'


In [81]:
pr_server.terminate()

In [82]:
client("qwerty")

start client data='qwerty'


ConnectionRefusedError: [Errno 61] Connection refused

In [83]:
pr_server.is_alive()

False

In [92]:
def client(data):
    print(f"start client {data=}")
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.connect(("localhost", 20_001))
        sock.sendall(data.encode())
        updated = sock.recv(1024).decode()
        print(f"client recv {updated=}")

    print(f"finish client {data=}")


def server():
    print(f"start server")

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("localhost", 20_001))
        sock.listen(5)

        while True:
            client, addr = sock.accept()
            print(f"server accepted {client=}, {addr=}")

            data = client.recv(1024)
            data = data.decode()
            print(f"recv {data=}")

            client.sendall(data.upper().encode())

    print(f"finish server")


pr_server = ctx.Process(target=server)
pr_server.start()

start server
server accepted client=<socket.socket fd=26, family=2, type=1, proto=0, laddr=('127.0.0.1', 20001), raddr=('127.0.0.1', 57637)>, addr=('127.0.0.1', 57637)
recv data='qwerty'


In [93]:
client("qwerty")

start client data='qwerty'
client recv updated='QWERTY'
finish client data='qwerty'


In [94]:
pr_server.name, pr_server.pid

('ForkProcess-74', 74784)

In [95]:
!ps -ef | grep 74784

  502 74784 35355   0  8:49   ??         0:00.02 /Library/Frameworks/Python.framework/Versions/3.14/Resources/Python.app/Contents/MacOS/Python -m ipykernel_launcher -f /Users/g.kandaurov/Library/Jupyter/runtime/kernel-78db0713-a525-49f1-a8dd-beb4f1591284.json
  502 75353 35355   0  8:49   ttys015    0:00.02 /bin/zsh -c ps -ef | grep 74784
  502 75355 75353   0  8:49   ttys015    0:00.00 grep 74784


In [97]:
!kill -9 74784

In [96]:
pr_server.is_alive()

True

In [98]:
pr_server.is_alive()

False

In [99]:
!ps -ef | grep 74784

  502 77677 35355   0  8:49   ttys015    0:00.02 /bin/zsh -c ps -ef | grep 74784
  502 77679 77677   0  8:49   ttys015    0:00.00 grep 74784


In [100]:
import signal

In [118]:
def client(data):
    print(f"start client {data=}")
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.connect(("localhost", 20_002))
        sock.sendall(data.encode())
        updated = sock.recv(1024).decode()
        print(f"client recv {updated=}")

    print(f"finish client {data=}")


def server():
    def process_sigterm(*args, **kwargs):
        print(f"process_sigterm {args=}, {kwargs=}")
    
    signal.signal(signal.SIGTERM, process_sigterm)

    print(f"start server")

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEPORT, 1)
        sock.bind(("localhost", 20_002))
        sock.listen(5)

        while True:
            client, addr = sock.accept()
            print(f"server accepted {client=}, {addr=}")

            data = client.recv(1024)
            data = data.decode()
            print(f"recv {data=}")

            client.sendall(data.upper().encode())

    print(f"finish server")


pr_server = ctx.Process(target=server)
pr_server.start()

start server
server accepted client=<socket.socket fd=26, family=2, type=1, proto=0, laddr=('127.0.0.1', 20002), raddr=('127.0.0.1', 58318)>, addr=('127.0.0.1', 58318)
recv data='qwerty'
process_sigterm args=(15, <frame at 0x109729e40, file '/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/socket.py', line 298, code accept>), kwargs={}


In [119]:
client("qwerty")

start client data='qwerty'
client recv updated='QWERTY'
finish client data='qwerty'


In [120]:
pr_server.is_alive()

True

In [121]:
pr_server.pid

15971

In [122]:
!kill -15 15971

In [123]:
pr_server.is_alive()

True

In [124]:
!kill -9 15971

In [ ]:
pr_server.is_alive()